# How to run this notebook

This notebook mirrors the shell-based workflow described in `README.md`: create the PostgreSQL schema, import the cafeteria dataset, and export the normalized tables back to CSV.

Before running the cells:

1. Install dependencies with `poetry install`.
2. Create `.env` from `.env.example` and fill in `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER`, and `PGPASSWORD`.
3. Start PostgreSQL with `docker compose up -d db`.
4. Open this notebook from the repository root and use the project Python environment as the kernel.

If your notebook kernel already points to the project environment, the `!python -m ...` commands below will run correctly. If not, replace them with `!poetry run python -m ...`. Run the cells from top to bottom because each step depends on the previous one.

## 1. Create database tables
Run `python -m src.init_database` to initialize the relational schema. This command reads `src/sql/init.sql` and creates the three main PostgreSQL tables used in the project: `clientes`, `visitas_mensuales`, and `compra_restaurante`.

Use this cell first, before importing any data, so the target tables already exist.

In [11]:
!python -m src.init_database

[23:27:43] INFO     Initializing database...                init_database.py:12
[23:27:43] DEBUG    File found at 'src\sql\init.sql'                files.py:57
[23:27:43] DEBUG    File at 'src\sql\init.sql' read successfully    files.py:41
                    with 353 characters                                        
┌───────────────────────────── Commands Executed ─────────────────────────────┐
│    1 CREATE TABLE IF NOT EXISTS clientes (                                  │
│    2     id int PRIMARY KEY,                                                │
│    3     edad float,                                                        │
│    4     genero text,                                                       │
│    5     fecha_registro date                                                │
│    6 );                                                                     │
│    7                                                                        │
│    8 CREATE TABLE IF NOT EXISTS visita

## 2. Import the source dataset
Run `python -m src.import_dataset` to load `src/data/dataset_clientes_cafeteria.csv` into PostgreSQL. The script first creates a temporary staging table, then copies the raw CSV into that table, and finally inserts normalized values into the three final tables.

Use this cell after the schema has been created successfully.

In [12]:
!python -m src.import_dataset

[23:27:46] INFO     Importing dataset script...            import_dataset.py:10
[23:27:46] DEBUG    File found at 'src\sql\temp_stage_table.sql'    files.py:57
[23:27:46] DEBUG    File at 'src\sql\temp_stage_table.sql' read     files.py:41
                    successfully with 187 characters                           
┌───────────────────────────── Commands Executed ─────────────────────────────┐
│   1 CREATE TEMP TABLE stage_clientes (                                      │
│   2     cliente_id text,                                                    │
│   3     edad text,                                                          │
│   4     genero text,                                                        │
│   5     monto_compra text,                                                  │
│   6     metodo_pago text,                                                   │
│   7     visitas_mensuales text,                                             │
│   8     fecha_registro text           

## 3. Export the normalized tables
Run `python -m src.export_database` to export the cleaned relational tables back to CSV files. This command writes one file per table under `src/data/export/`: `clientes.csv`, `visitas_mensuales.csv`, and `compra_restaurante.csv`.

Use this final cell after the import step if you want to verify the resulting outputs or reuse them outside PostgreSQL.

In [13]:
!python -m src.export_database

[23:27:48] INFO     Exporting table 'clientes' to CSV...  export_database.py:11
┌─────────────────────────── COPY Command Executed ───────────────────────────┐
│   1 COPY clientes TO STDOUT WITH CSV HEADER                                 │
└─────────────────────────────────────────────────────────────────────────────┘
[23:27:48] INFO     Data exported successfully to                     sql.py:95
                    'src\data\export\clientes.csv'                             
[23:27:48] INFO     Table 'clientes' exported             export_database.py:16
                    successfully to                                            
                    'src\data\export\clientes.csv'                             
[23:27:48] INFO     Exporting table 'visitas_mensuales'   export_database.py:11
                    to CSV...                                                  
┌─────────────────────────── COPY Command Executed ───────────────────────────┐
│   1 COPY visitas_mensuales TO STDOUT W